In [66]:
import json
import urllib.request
import zipfile

import geopandas as gpd
import matplotlib.patheffects as pe
from matplotlib.patches import Circle, Wedge
from shapely.geometry import box

CLUSTER_MAP_BEM_PATH = Path("../src/data/BEM/rbsa_sites.json")
CLUSTER_MAP_GEO_DIR = Path("../src/data/geo")
CLUSTER_MAP_STATE_URL = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_20m.zip"
CLUSTER_MAP_ZCTA_URL = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_zcta520_500k.zip"
CLUSTER_MAP_PROJ_CRS = "EPSG:5070"


def normalize_cluster_map_site_id(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


def normalize_cluster_map_zip(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().split("-", 1)[0]
    if text.endswith(".0"):
        text = text[:-2]
    return text.zfill(5) if text else np.nan


def load_cluster_map_census_shapefile(url, cache_dir):
    cache_dir.mkdir(parents=True, exist_ok=True)
    zip_path = cache_dir / url.rsplit("/", 1)[-1]
    extract_dir = cache_dir / zip_path.stem

    if not zip_path.exists():
        print(f"Downloading {zip_path.name}...")
        with urllib.request.urlopen(url, timeout=60) as response:
            zip_path.write_bytes(response.read())

    if not any(extract_dir.glob("*.shp")):
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)

    return gpd.read_file(next(extract_dir.glob("*.shp")))


def draw_cluster_ratio_pie(ax, x, y, counts, radius, colors):
    total = counts.sum()
    if total <= 0:
        return

    start = 90
    for cluster_id, count in counts[counts > 0].items():
        sweep = 360 * count / total
        ax.add_patch(
            Wedge(
                (x, y),
                radius,
                start,
                start + sweep,
                facecolor=colors[cluster_id],
                edgecolor="white",
                linewidth=0.4,
                zorder=5,
            )
        )
        start += sweep

    ax.add_patch(
        Circle(
            (x, y),
            radius,
            facecolor="none",
            edgecolor="#222222",
            linewidth=0.5,
            zorder=6,
        )
    )


with open(CLUSTER_MAP_BEM_PATH, encoding="utf-8") as f:
    rbsa_sites_for_map = json.load(f)

bem_location = pd.DataFrame(
    [
        {
            "ee_site_id": normalize_cluster_map_site_id(site.get("info", {}).get("ee_site_id")),
            "bem_city": site.get("info", {}).get("city"),
            "bem_county": site.get("info", {}).get("county"),
            "bem_state": site.get("info", {}).get("state"),
            "bem_zip": normalize_cluster_map_zip(site.get("info", {}).get("zip")),
        }
        for site in rbsa_sites_for_map
    ]
).drop_duplicates("ee_site_id")

cluster_map_assignments = weekly_summary_cluster_assignments.reset_index().rename(columns={"index": "ee_site_id"})
cluster_map_assignments["ee_site_id"] = cluster_map_assignments["ee_site_id"].map(normalize_cluster_map_site_id)

cluster_map_sites = cluster_map_assignments.merge(bem_location, on="ee_site_id", how="left")
cluster_map_missing_zip = cluster_map_sites["bem_zip"].isna().sum()
print(
    f"Mapped {len(cluster_map_sites) - cluster_map_missing_zip:,} / {len(cluster_map_sites):,} clustered sites "
    f"using ZIP codes from {CLUSTER_MAP_BEM_PATH.name}."
)

display(
    cluster_map_sites.groupby(["weekly_summary_cluster", "bem_state", "bem_county", "bem_zip"])
    .size()
    .rename("n_sites")
    .reset_index()
    .sort_values(["weekly_summary_cluster", "bem_state", "bem_county", "n_sites"], ascending=[True, True, True, False])
)

zip_cluster_counts = pd.crosstab(
    cluster_map_sites["bem_zip"],
    cluster_map_sites["weekly_summary_cluster"],
).sort_index()
zip_cluster_counts["n_sites"] = zip_cluster_counts.sum(axis=1)

states = load_cluster_map_census_shapefile(CLUSTER_MAP_STATE_URL, CLUSTER_MAP_GEO_DIR)
states = states[~states["STUSPS"].isin(["AK", "HI", "PR", "VI", "GU", "MP", "AS"])].to_crs(CLUSTER_MAP_PROJ_CRS)

zctas = load_cluster_map_census_shapefile(CLUSTER_MAP_ZCTA_URL, CLUSTER_MAP_GEO_DIR)
zcta_zip_column = next(column for column in zctas.columns if column.startswith("ZCTA5CE"))
zctas = zctas[[zcta_zip_column, "geometry"]].rename(columns={zcta_zip_column: "bem_zip"})
zctas["bem_zip"] = zctas["bem_zip"].map(normalize_cluster_map_zip)
zctas = zctas.to_crs(CLUSTER_MAP_PROJ_CRS)

zip_cluster_geo = zctas.merge(zip_cluster_counts.reset_index(), on="bem_zip", how="inner")
zip_cluster_geo["plot_point"] = zip_cluster_geo.geometry.representative_point()
zip_cluster_geo["x"] = zip_cluster_geo["plot_point"].x
zip_cluster_geo["y"] = zip_cluster_geo["plot_point"].y

missing_zcta_zips = sorted(set(zip_cluster_counts.index) - set(zip_cluster_geo["bem_zip"]))
if missing_zcta_zips:
    print(f"WARNING: {len(missing_zcta_zips)} ZIP(s) had no matching Census ZCTA geometry: {missing_zcta_zips[:10]}")

if zip_cluster_geo.empty:
    raise ValueError("No clustered ZIP codes matched Census ZCTA geometries.")

cluster_ids = sorted(cluster_map_sites["weekly_summary_cluster"].dropna().unique())
cluster_colors = {cluster_id: plt.get_cmap("tab10")(i % 10) for i, cluster_id in enumerate(cluster_ids)}

fig, ax = plt.subplots(figsize=(11, 8))
states.plot(ax=ax, color="#f6f4ee", edgecolor="#aaaaaa", linewidth=0.6, zorder=1)
zip_cluster_geo.boundary.plot(ax=ax, color="#dddddd", linewidth=0.25, alpha=0.45, zorder=2)

pad = 90_000
x_min, y_min, x_max, y_max = zip_cluster_geo.total_bounds
view_poly = box(x_min - pad, y_min - pad, x_max + pad, y_max + pad)
data_states = set(cluster_map_sites["bem_state"].dropna().unique())
for _, row in states[states["STUSPS"].isin(data_states)].iterrows():
    clipped = row.geometry.intersection(view_poly)
    if clipped.is_empty:
        continue
    ax.text(
        clipped.centroid.x,
        clipped.centroid.y,
        row["STUSPS"],
        ha="center",
        va="center",
        fontsize=8,
        color="#333333",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
        zorder=4,
    )

max_zip_sites = zip_cluster_geo["n_sites"].max()
cluster_columns = [column for column in zip_cluster_counts.columns if column != "n_sites"]
for _, row in zip_cluster_geo.iterrows():
    radius = 10_000 + 35_000 * np.sqrt(row["n_sites"] / max_zip_sites)
    draw_cluster_ratio_pie(ax, row["x"], row["y"], row[cluster_columns], radius, cluster_colors)

for _, row in zip_cluster_geo.sort_values("n_sites", ascending=False).head(12).iterrows():
    ax.text(
        row["x"],
        row["y"] + 48_000,
        f"{row['bem_zip']} ({int(row['n_sites'])})",
        ha="center",
        va="bottom",
        fontsize=7,
        color="#222222",
        path_effects=[pe.withStroke(linewidth=2, foreground="white")],
        zorder=7,
    )

cluster_handles = [
    Wedge((0, 0), 1, 0, 360, facecolor=cluster_colors[cluster_id], edgecolor="white", label=f"Cluster {cluster_id}")
    for cluster_id in cluster_ids
]
cluster_legend = ax.legend(handles=cluster_handles, title="Pie slice", loc="upper right", framealpha=0.92)
ax.add_artist(cluster_legend)

for val in sorted({1, int(np.ceil(max_zip_sites / 2)), int(max_zip_sites)}):
    radius = 10_000 + 35_000 * np.sqrt(val / max_zip_sites)
    ax.scatter(
        [],
        [],
        s=(radius / 900) ** 2,
        facecolor="none",
        edgecolor="#222222",
        label=f"{val} site{'s' if val != 1 else ''}",
    )
ax.legend(title="Pie size", loc="lower right", framealpha=0.92)

ax.set_xlim(x_min - pad, x_max + pad)
ax.set_ylim(y_min - pad, y_max + pad)
ax.set_title("Weekly load cluster ratios by ZIP code")
ax.set_axis_off()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'geopandas'